# 03 - Limpieza de Datos

**Objetivo**: Limpiar datos crudos, manejar valores faltantes, normalizar formatos.

Pasos:
1. Cargar datos de SQLite
2. Identificar valores faltantes y outliers
3. Limpiar y normalizar
4. Guardar datos limpios como Parquet

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data.storage import Storage
import config

storage = Storage()
print("Estado de la BD:")
for table, count in storage.stats().items():
    print(f"  {table}: {count} registros")

## 1. Cargar y examinar datos

In [ ]:
# Cargar todas las tablas principales
tokens_df = storage.query("SELECT * FROM tokens")
snapshots_df = storage.query("SELECT * FROM pool_snapshots")
ohlcv_df = storage.query("SELECT * FROM ohlcv")
holders_df = storage.query("SELECT * FROM holder_snapshots")
contracts_df = storage.query("SELECT * FROM contract_info")

print(f"Tokens: {len(tokens_df)} filas, {tokens_df.columns.tolist()}")
print(f"Snapshots: {len(snapshots_df)} filas")
print(f"OHLCV: {len(ohlcv_df)} filas")
print(f"Holders: {len(holders_df)} filas")
print(f"Contratos: {len(contracts_df)} filas")

In [ ]:
# Examinar los primeros registros de tokens
if not tokens_df.empty:
    display(tokens_df.head())
    print(f"\nValores nulos por columna:")
    print(tokens_df.isnull().sum())
else:
    print("No hay tokens en la BD. Ejecuta notebook 02 primero.")

## 2. Identificar problemas de calidad

In [ ]:
# Verificar tipos de datos
if not tokens_df.empty:
    print("Tipos de datos en tokens:")
    print(tokens_df.dtypes)
    
    # Tokens duplicados?
    dupes = tokens_df['token_id'].duplicated().sum()
    print(f"\nTokens duplicados: {dupes}")
    
    # Tokens sin nombre
    no_name = tokens_df['name'].isna().sum()
    print(f"Tokens sin nombre: {no_name}")
    
    # Tokens sin pool
    no_pool = tokens_df['pool_address'].isna().sum()
    print(f"Tokens sin pool_address: {no_pool}")

In [ ]:
# Examinar OHLCV - buscar anomalias
if not ohlcv_df.empty:
    print("Estadisticas OHLCV:")
    print(ohlcv_df[['open', 'high', 'low', 'close', 'volume']].describe())
    
    # Velas con precio 0?
    zero_price = (ohlcv_df['close'] == 0).sum()
    print(f"\nVelas con close = 0: {zero_price}")
    
    # Velas con volumen 0?
    zero_vol = (ohlcv_df['volume'] == 0).sum()
    print(f"Velas con volume = 0: {zero_vol}")
    
    # High < Low? (error de datos)
    bad_candles = (ohlcv_df['high'] < ohlcv_df['low']).sum()
    print(f"Velas con high < low (error): {bad_candles}")
else:
    print("No hay datos OHLCV")

In [ ]:
# Examinar snapshots - buscar valores extremos
if not snapshots_df.empty:
    print("Estadisticas de snapshots:")
    numeric_cols = ['price_usd', 'volume_24h', 'liquidity_usd', 'market_cap', 'fdv']
    existing_cols = [c for c in numeric_cols if c in snapshots_df.columns]
    print(snapshots_df[existing_cols].describe())
    
    # Valores negativos (no deberian existir)
    for col in existing_cols:
        negatives = (snapshots_df[col] < 0).sum()
        if negatives > 0:
            print(f"\n⚠️ {col}: {negatives} valores negativos")
else:
    print("No hay snapshots")

## 3. Limpieza

In [ ]:
# --- Limpieza de OHLCV ---
if not ohlcv_df.empty:
    ohlcv_clean = ohlcv_df.copy()
    initial_len = len(ohlcv_clean)
    
    # 1. Eliminar velas con precio 0 o negativo
    ohlcv_clean = ohlcv_clean[ohlcv_clean['close'] > 0]
    
    # 2. Corregir high < low (swap)
    mask = ohlcv_clean['high'] < ohlcv_clean['low']
    ohlcv_clean.loc[mask, ['high', 'low']] = ohlcv_clean.loc[mask, ['low', 'high']].values
    
    # 3. Eliminar duplicados
    ohlcv_clean = ohlcv_clean.drop_duplicates(
        subset=['pool_address', 'timeframe', 'timestamp']
    )
    
    # 4. Convertir timestamp a datetime
    ohlcv_clean['timestamp'] = pd.to_datetime(ohlcv_clean['timestamp'])
    
    # 5. Ordenar por token y tiempo
    ohlcv_clean = ohlcv_clean.sort_values(['token_id', 'timeframe', 'timestamp'])
    
    removed = initial_len - len(ohlcv_clean)
    print(f"OHLCV: {initial_len} -> {len(ohlcv_clean)} ({removed} eliminadas)")
else:
    ohlcv_clean = pd.DataFrame()

In [ ]:
# --- Limpieza de snapshots ---
if not snapshots_df.empty:
    snaps_clean = snapshots_df.copy()
    initial_len = len(snaps_clean)
    
    # 1. Eliminar filas sin token_id
    snaps_clean = snaps_clean.dropna(subset=['token_id'])
    
    # 2. Reemplazar valores negativos con NaN
    numeric_cols = ['price_usd', 'volume_24h', 'liquidity_usd', 'market_cap', 'fdv']
    for col in numeric_cols:
        if col in snaps_clean.columns:
            snaps_clean.loc[snaps_clean[col] < 0, col] = np.nan
    
    # 3. Convertir timestamps
    snaps_clean['snapshot_time'] = pd.to_datetime(snaps_clean['snapshot_time'])
    
    removed = initial_len - len(snaps_clean)
    print(f"Snapshots: {initial_len} -> {len(snaps_clean)} ({removed} eliminados)")
else:
    snaps_clean = pd.DataFrame()

In [ ]:
# --- Limpieza de tokens ---
if not tokens_df.empty:
    tokens_clean = tokens_df.copy()
    
    # 1. Eliminar tokens sin pool_address (no podemos obtener datos)
    tokens_clean = tokens_clean.dropna(subset=['pool_address'])
    
    # 2. Normalizar nombres de cadena
    tokens_clean['chain'] = tokens_clean['chain'].str.lower().str.strip()
    
    # 3. Convertir timestamps
    tokens_clean['created_at'] = pd.to_datetime(tokens_clean['created_at'], errors='coerce')
    tokens_clean['first_seen'] = pd.to_datetime(tokens_clean['first_seen'], errors='coerce')
    
    print(f"Tokens limpios: {len(tokens_clean)}")
    print(f"Por cadena: {tokens_clean['chain'].value_counts().to_dict()}")
else:
    tokens_clean = pd.DataFrame()

## 4. Guardar datos limpios como Parquet

In [ ]:
# Guardar en formato Parquet (mas eficiente que CSV)
output_dir = config.PROCESSED_DIR

if not tokens_clean.empty:
    tokens_clean.to_parquet(output_dir / "tokens_clean.parquet", index=False)
    print(f"Guardado: tokens_clean.parquet ({len(tokens_clean)} filas)")

if not ohlcv_clean.empty:
    ohlcv_clean.to_parquet(output_dir / "ohlcv_clean.parquet", index=False)
    print(f"Guardado: ohlcv_clean.parquet ({len(ohlcv_clean)} filas)")

if not snaps_clean.empty:
    snaps_clean.to_parquet(output_dir / "snapshots_clean.parquet", index=False)
    print(f"Guardado: snapshots_clean.parquet ({len(snaps_clean)} filas)")

if not holders_df.empty:
    holders_df.to_parquet(output_dir / "holders_clean.parquet", index=False)
    print(f"Guardado: holders_clean.parquet ({len(holders_df)} filas)")

if not contracts_df.empty:
    contracts_df.to_parquet(output_dir / "contracts_clean.parquet", index=False)
    print(f"Guardado: contracts_clean.parquet ({len(contracts_df)} filas)")

print("\n✅ Datos limpios guardados en data/processed/")
print("Siguiente paso: notebook 04_eda.ipynb")